In [27]:
import sys
print(sys.executable)

c:\Users\EL077\Desktop\git-projects\Azure-OpenAI-kma-Weather-bot\.venv\Scripts\python.exe


In [31]:
# =============================
# 기본 세팅: 상수처럼 파일 상단에 둔다. 
# =============================
# 1. 라이브러리 설치
import os
import base64
import json
import requests
import time
from openai import AzureOpenAI
from dotenv import load_dotenv 

load_dotenv()



True

In [54]:
LOCATION_COORDS = {
    "서울": (37.5665, 126.9780),
    "부산": (35.1796, 129.0756),
    "대구": (35.8714, 128.6014),
    "인천": (37.4563, 126.7052),
    "광주": (35.1595, 126.8526),
    "대전": (36.3504, 127.3845),
    "울산": (35.5384, 129.3114),
    "세종": (36.4800, 127.2890),
    "제주": (33.4996, 126.5312),
}

# 1. get_korea_weather 함수

In [38]:
from datetime import datetime, timedelta

# 강수 형태 매핑을 위한 딕셔너리
PTY_CODE = {
    "0": "강수 없음",
    "1": "비",
    "2": "비/눈",
    "3": "눈",
    "5": "빗방울",
    "6": "빗방울/눈날림",
    "7": "눈날림"
}

# ===============================
# [1] 기상청 초단기 실황 api에 넣을 기준 날짜와 기준 시간을 자동으로 만들어주는 함수 
# ===============================
def get_base_time_for_ncst():
    now = datetime.now()

    # 초단기실황은 정시 기준 자료라, 안정적인 데이터 로드를 위해 너무 이른 시각이면 이전 시간 사용
    if now.minute < 40:
        now = now - timedelta(hours=1)

    return now.strftime("%Y%m%d"), now.strftime("%H00")


# ===============================
# [2] 위도, 경도를 기상청 전용 격자 좌표로 바꿔주는 함수
# ===============================
def convert_lat_lon_to_grid(lat, lon):
    # 지구 반지름
    RE = 6371.00877

    # 격자 간격 5km
    GRID = 5.0

    # 투영 계산에 쓰는 기준 위도
    SLAT1 = 30.0
    SLAT2 = 60.0

    # 기준 경도
    OLON = 126.0

    # 기준 위도
    OLAT = 38.0

    # 기준점의 격자 좌표 보정값
    XO = 43
    YO = 136

    # 도 단위를 라디언으로 바꿈
    DEGRAD = math.pi / 180.0

    # 지구 반지름을 격자 단위로 환산 -> 지구 반지름 km /  격자 간격 5km
    re = RE / GRID

    # 투영 계산용 보정값 (지구는 둥그니까 지도로 펼칠 때 생기는 왜곡 보정)
    slat1 = SLAT1 * DEGRAD
    slat2 = SLAT2 * DEGRAD
    olon = OLON * DEGRAD
    olat = OLAT * DEGRAD

    sn = math.tan(math.pi * 0.25 + slat2 * 0.5) / math.tan(math.pi * 0.25 + slat1 * 0.5)
    sn = math.log(math.cos(slat1) / math.cos(slat2)) / math.log(sn)

    sf = math.tan(math.pi * 0.25 + slat1 * 0.5)
    sf = (sf ** sn) * math.cos(slat1) / sn

    ro = math.tan(math.pi * 0.25 + olat * 0.5)
    ro = re * sf / (ro ** sn)

    # 입력된 위도/경도를 위치로 변환
    ra = math.tan(math.pi * 0.25 + lat * DEGRAD * 0.5)
    ra = re * sf / (ra ** sn)

    # 경도 차이 계산
    theta = lon * DEGRAD - olon

    if theta > math.pi:
        theta -= 2.0 * math.pi
    if theta < -math.pi:
        theta += 2.0 * math.pi

    theta *= sn

    x = int(ra * math.sin(theta) + XO + 0.5)
    y = int(ro - ra * math.cos(theta) + YO + 0.5)

    return x, y


# ===============================
#  [3] 날씨 요청 함수
# ===============================
def get_korea_weather(location=None, latitude=None, longitude=None):

    # 환경변수에서 API 키와 요청 URL을 로드하고, 위경도를 기상청 격자 좌표로 변환
    weather_api_key = os.getenv("WEATHER_API_KEY")
    url = os.getenv("GET_KOREA_WEATHER")
    nx, ny = convert_lat_lon_to_grid(latitude, longitude)

    # 기상청 초단기실황 조회 기준일자 및 기준시각 계산
    base_date, base_time = get_base_time_for_ncst()
    
    # 기상청 API 요청 파라미터 구성
    params = {
        "serviceKey": weather_api_key,
        "pageNo": 1,
        "numOfRows": 100,
        "dataType": "JSON",
        "base_date": base_date,
        "base_time": base_time,
        "nx": nx,
        "ny": ny
    }

    # 기상청 API에 GET 요청 전송  
    response = requests.get(url, params=params)
    print("status_code:", response.status_code)
    # print("response preview:", response.text[:300]) # response의 모든 정보 

    # JSON 응답 파싱
    data = response.json()

    # 응답 헤더를 확인하여 API 처리 결과 검증
    header = data["response"]["header"]
    if header["resultCode"] != "00":
        return f"기상청 API 오류: {header['resultCode']} / {header['resultMsg']}"

    # 응답 본문에서 관측 항목 리스트 추출
    items = data["response"]["body"]["items"]["item"]

    # category 코드를 key로, obsrValue를 value로 매핑
    weather_info = {}
    for item in items:
        weather_info[item["category"]] = item["obsrValue"]

    # 기온
    temp = weather_info.get("T1H")  

    # 습도
    humidity = weather_info.get("REH")  

    # 강수형태
    rain_type = PTY_CODE.get(weather_info.get("PTY"), weather_info.get("PTY"))  

    # 1시간 강수량
    rain_1h = weather_info.get("RN1")  

    # 풍속
    wind_speed = weather_info.get("WSD")  
    
    # 사용자에게 전달할 날씨 응답 문자열 생성
    return (
        f"{location}의 기상청 초단기실황입니다. "
        f"기온은 {temp}℃, 습도는 {humidity}%, 강수형태는 {rain_type}, "
        f"1시간 강수량은 {rain_1h}mm, 풍속은 {wind_speed}m/s입니다. "
        f"조회 기준시각은 {base_date} {base_time}, 격자좌표는 nx={nx}, ny={ny}입니다."
    )

In [41]:
demo_weather = get_korea_weather(
    location="부산",
    latitude=35.1796,
    longitude=129.0756)
print(demo_weather)

status_code: 200
부산의 기상청 초단기실황입니다. 기온은 18.5℃, 습도는 46%, 강수형태는 강수 없음, 1시간 강수량은 0mm, 풍속은 1.7m/s입니다. 조회 기준시각은 20260606 0100, 격자좌표는 nx=98, ny=76입니다.


#  2. require_upgrade_chat 함수

### 1버전

In [39]:
################ 2버전 requset_gpt ####################
# 사용자로부터 텍스트로 입력받으면, 텍스트로 답변을 출력하는 챗봇 함수
def request_upgrade_gpt(prompt):

    # 환경 변수에서 엔드포인트, 배포 이름, 구독 키 가져오기
    endpoint = os.getenv("ENDPOINT_URL")
    deployment = os.getenv("DEPLOYMENT_NAME")
    subscription_key = os.getenv("AZURE_OPENAI_API_KEY")
    api_version= os.getenv("OPEN_API_VERSION")

    #-------------------
    # 메시지 개수 제한하는 코드 추가 - 최근 messages_limit개만 기억하도록
    #-------------------
    messages_cnt = 3
    messages_limit = messages_cnt*2

    #-------------------
    # Initialize Azure OpenAI client with key-based authentication
    #-------------------
    client = AzureOpenAI(
        azure_endpoint=endpoint,
        api_key=subscription_key,
        api_version=api_version
    )

    #------------------
    # Prepare the chat prompt
    #------------------
    chat_prompt = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": "너는 날씨정보를 기상청으로부터 받아와서 사용자에게 알려주는 챗봇이야."
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }]
    
    #--------------
    #  Include speech result if speech is enabled
    #--------------
    messages = chat_prompt

    #-------------
    # 사용자가 '그만'이라고 할 때까지 챗봇이 계속 대화할 수 있도록 while문으로 만들어보자
    #-------------
    while True:
        
        # 메시지 개수 제한하는 코드 추가 - 최근 messages_limit개만 기억하도록
        if len(messages) > (1+messages_limit):
            messages = [messages[0]] + messages[-messages_limit:]
            # messages = messages[0] + messages[-messages_limit:] -> dict + list 라서 에러 뜸. messages[0]는 딕셔너리이므로 []로 한번 더 감싸줘야함. 즉 [system 메시지] + [최근 대화 6개]로 해야함. 
        
        # 사용자로부터 입력 받기
        user_input = input("user: ")
        
        # 사용자가 '그만'이라고 하면 챗봇과의 대화를 종료한다.
        if user_input.lower() in ["그만", "그만해", "종료", "끝", "stop"]:
            print("챗봇과의 대화를 종료합니다.")
            break
        print("user: " + user_input)
        messages.append(
            {"role": "user",
            "content": [{"type": "text", "text": user_input}]
            }
        )
        #------------
        # Generate the completion
        #------------
        completion = client.chat.completions.create(
            model=deployment,
            messages=messages,
            max_tokens=6553,
            temperature=0.7,
            top_p=0.95,
            frequency_penalty=0,
            presence_penalty=0,
            stop=None,
            stream=False
        )

    #----------------
    #  response를 변수로 저장해서, messages에 assistant 역할로 추가해준다. 그래야 다음 대화에서도 이전 답변을 기억할 수 있다.
    #----------------
    response = completion.choices[0].message.content
    # baselien -> 최근 6개까지만 기억하는 것 성공
    # messages.append(
    #     {"role": "assistant",
    #     "content": [{"type": "text", "text": response}]
    #     }
    # )
    if "날씨" in user_input:
        weather_result = get_korea_weather(
            location="부산",
            latitude=35.1796,
            longitude=129.0756
        )

        user_message = f"""
    사용자 질문: {user_input}

    아래는 기상청 API 조회 결과야.
    이 정보를 바탕으로 사용자에게 자연스럽게 답변해줘.

    {weather_result}
    """
    else:
        user_message = user_input

        messages.append(
            {
                "role": "user",
                "content": [{"type": "text", "text": user_message}]
            }
        )
    return completion.choices[0].message.content


In [42]:
# demo

demo_chat = request_upgrade_gpt('오늘의 서울 날씨 알려줘.')
print(demo_chat)

user: 현재 부산날씨 알려줘
챗봇과의 대화를 종료합니다.
죄송하지만, 실시간 날씨 정보를 제공할 수는 없습니다. 하지만 기상청 웹사이트나 날씨 앱을 통해 서울과 부산의 현재 날씨를 확인하실 수 있습니다. 도움이 필요하시면 다른 질문해 주세요!


### 2버전

In [55]:
def extract_location(user_input):
    for location in LOCATION_COORDS:
        if location in user_input:
            return location
    return None

In [ ]:
################ 2버전 requset_gpt ####################
# 사용자로부터 텍스트로 입력받으면, 텍스트로 답변을 출력하는 챗봇 함수
def generate_chat_response(prompt):

    # 환경 변수에서 엔드포인트, 배포 이름, 구독 키 가져오기
    endpoint = os.getenv("ENDPOINT_URL")
    deployment = os.getenv("DEPLOYMENT_NAME")
    subscription_key = os.getenv("AZURE_OPENAI_API_KEY")
    api_version= os.getenv("OPEN_API_VERSION")

    #-------------------
    # 메시지 개수 제한하는 코드 추가 - 최근 messages_limit개만 기억하도록
    #-------------------
    messages_cnt = 3
    messages_limit = messages_cnt*2

    #-------------------
    # Initialize Azure OpenAI client with key-based authentication
    #-------------------
    client = AzureOpenAI(
        azure_endpoint=endpoint,
        api_key=subscription_key,
        api_version=api_version
    )

    #--------------
    #  Include speech result if speech is enabled
    #--------------
    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": "너는 날씨정보를 기상청으로부터 받아와서 사용자에게 알려주는 챗봇이야."
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }]
    

    #-------------
    # 사용자가 '그만'이라고 할 때까지 챗봇이 계속 대화할 수 있도록 while문으로 만들어보자
    #-------------
    while True:
        
        # 메시지 개수 제한하는 코드 추가 - 최근 messages_limit개만 기억하도록
        if len(messages) > (1+messages_limit):
            messages = [messages[0]] + messages[-messages_limit:]
        
        # 사용자로부터 입력 받기
        user_input = input("user: ")
        
        # 사용자가 '그만'이라고 하면 챗봇과의 대화를 종료한다.
        if user_input.lower() in ["그만", "그만해", "종료", "끝", "stop"]:
            print("챗봇과의 대화를 종료합니다.")
            break
        print("user: " + user_input)

        # 날씨 질문이면 기상청 api와 함께 전달
        location = extract_location(user_input)
        if "날씨" in user_input and location:
            latitude, longitude = LOCATION_COORDS[location]

            weather_result = get_korea_weather(
                location=location,
                latitude=latitude,
                longitude=longitude
            )

            user_message = f"""
    사용자 질문: {user_input}

    아래는 기상청 API 조회 결과야.
    이 정보를 바탕으로 사용자에게 자연스럽게 답변해줘.

    {weather_result}
    """
        else:
            user_message = user_input

        messages.append(
            {
                "role": "user",
                "content": [{"type": "text", "text": user_message}]
            }
        )

        #------------
        # Generate the completion
        #------------
        completion = client.chat.completions.create(
            model=deployment,
            messages=messages,
            max_tokens=6553,
            temperature=0.7,
            top_p=0.95,
            frequency_penalty=0,
            presence_penalty=0,
            stop=None,
            stream=False
        )

        #----------------
        #  response를 변수로 저장해서, messages에 assistant 역할로 추가해준다. 그래야 다음 대화에서도 이전 답변을 기억할 수 있다.
        #----------------
        response = completion.choices[0].message.content
        print("bot: ", response)
        # baselien -> 최근 6개까지만 기억하는 것 성공
        messages.append(
            {"role": "assistant",
            "content": [{"type": "text", "text": response}]
            }
        )
        
    return "챗봇종료"


In [57]:
# demo

demo_chat = generate_chat_response('')
print(demo_chat)

user: 세종 날씨 알려줘
status_code: 200
bot:  세종의 현재 날씨를 알려드릴게요. 현재 기온은 13.4℃이고, 습도는 84%로 다소 습한 편입니다. 강수는 없고, 1시간 강수량도 0mm입니다. 바람은 시속 0.7m로 거의 없는 상태입니다. 다른 궁금한 점이 있으면 언제든지 물어보세요!
user: 대전 날씨 알려줘
status_code: 200
bot:  대전의 현재 날씨를 알려드릴게요. 현재 기온은 14.5℃이며, 습도는 76%로 비교적 쾌적한 상태입니다. 강수는 없고, 1시간 강수량도 0mm입니다. 바람은 시속 1.8m로 약하게 불고 있습니다. 더 궁금한 점이 있으시면 언제든지 말씀해 주세요!
챗봇과의 대화를 종료합니다.
챗봇종료
